# Assignment 2 - Predicting tumor type

In [1]:
# import my own modules
from data_representation import DataTable
from predictors import Predictor, MajorityClassPredictor, NearestNeighborPredictor
from exceptions import OutOfSampleError
from main import main, calculate_accuracy

**Load the data**

In [2]:
historical = DataTable("data/cancer_historical.csv")
new_cases = DataTable("data/cancer_new_cases.csv")
labels = DataTable("data/cancer_new_cases_labels.csv")

# the assignment says 599, 100 and 100 rows
print(len(historical.data))
print(len(new_cases.data))
print(len(labels.data))

599
100
100


- this is the small test file, it only has 4 rows

In [3]:
file = open("data/test_small.csv", "r")
print(file.read())
file.close()

small = DataTable("data/test_small.csv")

Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,TumorType
1,1,1,1,1,1,1,1,1,0
4,5,1,1,1,1,1,1,1,1
10,10,10,10,10,10,10,10,10,1
2,?,3,3,3,3,3,3,3,1



**Task 1 - DataTable class**

**Normal cases**
- input: the small table, get_column_names
- expected: Feature1 to Feature9 and TumorType

In [4]:
print(small.get_column_names())

['Feature1', 'Feature2', 'Feature3', 'Feature4', 'Feature5', 'Feature6', 'Feature7', 'Feature8', 'Feature9', 'TumorType']


- input: get row 1 column Feature2, get row 1, get column Feature1
- expected: 5.0, the dictionary for row 1, and [1.0, 4.0, 10.0, 2.0]
- all the values should be floats

In [5]:
print(small.get(1, "Feature2"))
print(small.get_row(1))
print(small.get_column("Feature1"))

5.0
{'Feature1': 4.0, 'Feature2': 5.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 1.0, 'Feature6': 1.0, 'Feature7': 1.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 1.0}
[1.0, 4.0, 10.0, 2.0]


- input: Feature1, the values are 1, 4, 10, 2
- expected: average = 17 / 4 = 4.25, min = 1.0, max = 10.0

In [6]:
print(small.get_column_average("Feature1"))
print(small.get_column_min("Feature1"))
print(small.get_column_max("Feature1"))

4.25
1.0
10.0


- input: row 0 of the historical file (5,1,1,1,2,1,3,1,1,0)
- expected: the same numbers as floats
- input: min and max of every column
- expected: every feature goes from 1.0 to 10.0

In [7]:
print(historical.get_row(0))

for feature in historical.get_column_names():
    print(feature, historical.get_column_min(feature), historical.get_column_max(feature))

{'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}
Feature1 1.0 10.0
Feature2 1.0 10.0
Feature3 1.0 10.0
Feature4 1.0 10.0
Feature5 1.0 10.0
Feature6 1.0 10.0
Feature7 1.0 10.0
Feature8 1.0 10.0
Feature9 1.0 10.0
TumorType 0.0 1.0


**Special cases**

- some rows have a ? instead of a number (row 3 of the small file and row 23 of the historical file)
- ? can't be turned into a float so its stored as None 
- average, min and max skip the None values
- input: Feature2 of the small file, the values are 1, 5, 10, ?
- expected: the ? is None, average = 16 / 3 = 5.33, min = 1.0, max = 10.0

In [8]:
print(small.get_column("Feature2"))
print(small.get(3, "Feature2"))
print(small.get_column_average("Feature2"))
print(small.get_column_min("Feature2"))
print(small.get_column_max("Feature2"))

[1.0, 5.0, 10.0, None]
None
5.333333333333333
1.0
10.0


In [9]:
# row 23 of the historical file is 8,4,5,1,2,?,7,3,1,1
# expected: Feature6 is None, everything else is a float
print(historical.get_row(23))

{'Feature1': 8.0, 'Feature2': 4.0, 'Feature3': 5.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': None, 'Feature7': 7.0, 'Feature8': 3.0, 'Feature9': 1.0, 'TumorType': 1.0}


- input: first row (0) and last row (598) of the historical file
- expected: both work, row 598 is 3,1,1,1,2,1,2,1,1,0

In [10]:
print(historical.get_row(0))
print(historical.get_row(598))

{'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}
{'Feature1': 3.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 2.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}


**Error cases**

- input: a column that doesn't exist
- expected: ValueError

In [11]:
try:
    small.get_column("Feature99")
except ValueError as error:
    print("ValueError:", error)

ValueError: Feature99 is not a column in this table


- input: row 599 (rows only go from 0 to 598)
- expected: IndexError

In [12]:
try:
    historical.get_row(599)
except IndexError as error:
    print("IndexError:", error)

IndexError: row 599 does not exist, the table has 599 rows


- input: a file that doesn't exist
- expected: FileNotFoundError

In [13]:
try:
    DataTable("data/not_a_file.csv")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'data/not_a_file.csv'


**Task 2 - Predictor classes**

- Predictor is the abstract parent class
- MajorityClassPredictor and NearestNeighborPredictor are subclasses and each one has its own predict method
- make_patient makes a patient dictionary from a list of 9 numbers so the tests are shorter

In [14]:
def make_patient(values: list) -> dict:
    patient = {}
    for i in range(9):
        feature_name = "Feature" + str(i + 1)
        patient[feature_name] = values[i]
    return patient


print(make_patient([1, 2, 3, 4, 5, 6, 7, 8, 9]))

{'Feature1': 1, 'Feature2': 2, 'Feature3': 3, 'Feature4': 4, 'Feature5': 5, 'Feature6': 6, 'Feature7': 7, 'Feature8': 8, 'Feature9': 9}


**Mode prediction (MajorityClassPredictor)**

- input: the small file, its TumorTypes are 0, 1, 1, 1
- expected: 1 is the most common so every patient gets predicted 1

In [15]:
small_mode = MajorityClassPredictor(small)
print(small_mode.majority_class)

print(small_mode.predict(make_patient([1, 1, 1, 1, 1, 1, 1, 1, 1])))
print(small_mode.predict(make_patient([9, 9, 9, 9, 9, 9, 9, 9, 9])))

1
1
1


- input: the historical file
- expected: there are more benign (0) than malignant (1), so it predicts 0

In [16]:
# count each tumor type to check
benign = 0
malignant = 0
for tumor_type in historical.get_column("TumorType"):
    if tumor_type == 0:
        benign += 1
    else:
        malignant += 1

print("benign:", benign)
print("malignant:", malignant)

mode_predictor = MajorityClassPredictor(historical)
print("prediction:", mode_predictor.predict(new_cases.get_row(0)))

benign: 379
malignant: 220
prediction: 0


**Nearest neighbor prediction**

- input: distance between row 0 (1,1,1,1,1,1,1,1,1) and row 1 (4,5,1,1,1,1,1,1,1) of the small file
- expected: the differences are 3 and 4, so sqrt(9 + 16) = 5.0

In [17]:
small_nn = NearestNeighborPredictor(small)
print(small_nn.distance(small.get_row(0), small.get_row(1)))

5.0


- input: (1,1,1,1,1,1,1,1,2), expected: closest is row 0 so it predicts 0
- input: (4,4,1,1,1,1,1,1,1), expected: closest is row 1 so it predicts 1
- input: (9,9,9,9,9,9,9,9,9), expected: closest is row 2 so it predicts 1

In [18]:
print(small_nn.predict(make_patient([1, 1, 1, 1, 1, 1, 1, 1, 2])))
print(small_nn.predict(make_patient([4, 4, 1, 1, 1, 1, 1, 1, 1])))
print(small_nn.predict(make_patient([9, 9, 9, 9, 9, 9, 9, 9, 9])))

0
1
1


**Special cases**

- the distance only uses the 9 features, not TumorType
- input: row 0 and row 1 have different TumorTypes
- expected: still 5.0
- input: a patient compared to itself
- expected: 0.0

In [19]:
print(small_nn.distance(small.get_row(0), small.get_row(1)))
print(small_nn.distance(small.get_row(2), small.get_row(2)))

5.0
0.0


- input: a patient exactly the same as row 1 of the historical file (5,4,4,5,7,10,3,2,1), which is benign
- expected: its nearest neighbor is itself so it predicts 0

In [20]:
nn_predictor = NearestNeighborPredictor(historical)

patient = make_patient([5, 4, 4, 5, 7, 10, 3, 2, 1])
print(nn_predictor.predict(patient))

0


- input: row 18 of the new cases is 1,1,1,1,1,?,1,1,1
- expected: the missing feature gets skipped in the distance and it still gets a prediction

In [21]:
print(new_cases.get_row(18))
print(nn_predictor.predict(new_cases.get_row(18)))

{'Feature1': 1.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 1.0, 'Feature6': None, 'Feature7': 1.0, 'Feature8': 1.0, 'Feature9': 1.0}
0


- input: a patient with all 1s (the min) and a patient with all 10s (the max)
- expected: these are inside the range, so no OutOfSampleError

In [22]:
print(nn_predictor.predict(make_patient([1, 1, 1, 1, 1, 1, 1, 1, 1])))
print(nn_predictor.predict(make_patient([10, 10, 10, 10, 10, 10, 10, 10, 10])))

0
1


**Error case**

- input: making a Predictor object directly
- expected: TypeError because Predictor is abstract

In [23]:
try:
    Predictor(historical)
except TypeError as error:
    print("TypeError:", error)

TypeError: Can't instantiate abstract class Predictor without an implementation for abstract method 'predict'


**Task 3 - OutOfSampleError**

- a patient is too different if any feature is less than the min or more than the max of the historical data
- in the historical data every feature goes from 1.0 to 10.0
- input: Feature4 = 11
- expected: OutOfSampleError: Feature4 with value 11 is outside historical range [1.0, 10.0]

In [24]:
patient = make_patient([5, 1, 1, 11, 2, 1, 3, 1, 1])

try:
    nn_predictor.predict(patient)
except OutOfSampleError as error:
    print("OutOfSampleError:", error)

OutOfSampleError: Feature4 with value 11 is outside historical range [1.0, 10.0]


- input: Feature1 = 0
- expected: OutOfSampleError: Feature1 with value 0 is outside historical range [1.0, 10.0]

In [25]:
patient = make_patient([0, 1, 1, 1, 2, 1, 3, 1, 1])

try:
    nn_predictor.predict(patient)
except OutOfSampleError as error:
    print("OutOfSampleError:", error)

OutOfSampleError: Feature1 with value 0 is outside historical range [1.0, 10.0]


- input: the Feature4 = 11 patient again but with the mode predictor
- expected: OutOfSampleError, the mode predictor checks too

In [26]:
patient = make_patient([5, 1, 1, 11, 2, 1, 3, 1, 1])

try:
    mode_predictor.predict(patient)
except OutOfSampleError as error:
    print("OutOfSampleError:", error)

OutOfSampleError: Feature4 with value 11 is outside historical range [1.0, 10.0]


- input: the small file, Feature5 values are 1, 1, 10, 3 so the range is [1.0, 10.0], and the patient has Feature5 = 10.5
- expected: OutOfSampleError: Feature5 with value 10.5 is outside historical range [1.0, 10.0]

In [27]:
patient = make_patient([1, 1, 1, 1, 10.5, 1, 1, 1, 1])

try:
    small_nn.predict(patient)
except OutOfSampleError as error:
    print("OutOfSampleError:", error)

OutOfSampleError: Feature5 with value 10.5 is outside historical range [1.0, 10.0]


- this is the same way main.py handles the error, the bad patient gets skipped and the rest still get predicted
- input: 3 patients, the middle one has Feature9 = 12
- expected: patient 0 and 2 get a prediction and patient 1 prints the error

In [28]:
test_patients = []
test_patients.append(make_patient([1, 1, 1, 1, 2, 1, 1, 1, 1]))
test_patients.append(make_patient([1, 1, 1, 1, 2, 1, 1, 1, 12]))
test_patients.append(make_patient([10, 10, 10, 10, 10, 10, 10, 10, 10]))

for i in range(len(test_patients)):
    try:
        prediction = nn_predictor.predict(test_patients[i])
        print("Patient", i, "prediction:", prediction)
    except OutOfSampleError as error:
        print("Patient", i, "skipped. OutOfSampleError:", error)

Patient 0 prediction: 0
Patient 1 skipped. OutOfSampleError: Feature9 with value 12 is outside historical range [1.0, 10.0]
Patient 2 prediction: 1


**Task 4 - main program and accuracy**

**calculate_accuracy**

- input: predictions [1, 0, 1, 1] and actual [1, 0, 0, 1]
- expected: 3 out of 4 correct = 0.75

In [29]:
print(calculate_accuracy([1, 0, 1, 1], [1, 0, 0, 1]))

0.75


**Special cases**

- input: all correct, expected: 1.0
- input: all wrong, expected: 0.0

In [30]:
print(calculate_accuracy([1, 0, 1], [1, 0, 1]))
print(calculate_accuracy([1, 1, 1], [0, 0, 0]))

1.0
0.0


**Error cases**

- input: lists that are different lengths, expected: ValueError
- input: empty lists, expected: ValueError (so it doesn't divide by 0)

In [31]:
try:
    calculate_accuracy([1, 0], [1, 0, 1])
except ValueError as error:
    print("ValueError:", error)

try:
    calculate_accuracy([], [])
except ValueError as error:
    print("ValueError:", error)

ValueError: predictions and actual must be the same length
ValueError: there are no predictions to check


**Running main**

- main loads the 3 files, predicts all 100 new cases with both predictors, and compares them to the labels file
- expected: 100 predictions, no new case is out of sample because every feature is between 1 and 10
- the mode predictor predicts 0 for everyone, so its accuracy should be the same as the fraction of new cases that are benign
- nearest neighbor should be more accurate

In [32]:
main()

Number of predictions: 100
Mode accuracy: 0.79
Nearest Neighbor accuracy: 0.96


- checking the mode accuracy by hand: count how many of the true labels are 0

In [33]:
benign = 0
for tumor_type in labels.get_column("TumorType"):
    if tumor_type == 0:
        benign += 1

print("benign new cases:", benign, "out of", len(labels.data))

benign new cases: 79 out of 100


**Conclusion**

- mode prediction is right 79% of the time, but only because most tumors are benign, it never finds a malignant one
- nearest neighbor is right 96% of the time, so using similar past patients works a lot better